# 第七週：文件分類（PTT 伊朗議題）


## 目標
- 資料來源：PTT `Gossiping` 與 `Military` 兩個版別
- 研究主題：伊朗相關議題文章
- 任務：根據文章文字內容，預測文章屬於哪一個版別

## 主要步驟
1. 讀入資料來源 `ptt_iran_20260301_0410_full.csv`，並檢查版別分布與日期範圍
2. 清理文字、合併標題與內文、建立 `content`
3. 使用 jieba 斷詞並移除停用字，建立 `words`
4. 以 7:3 切分 train/test，建立 `CountVectorizer` 與 `TfidfVectorizer`
5. 先示範 Logistic Regression 的基本分類流程，再用 `classification_report` 與混淆矩陣評估
6. 使用 cross-validation 檢查模型穩定度
7. 比較多個分類器，選出表現最佳的模型
8. 用 Logistic Regression 係數觀察可解釋特徵
9. 使用 held-out test set 分析最佳模型的錯誤分類結果


## 1. 套件說明
- conda activate SMA2026
- `pandas`：整理表格資料與摘要統計
- `jieba`：中文斷詞
- `sklearn`：向量化、分類模型、cross-validation、模型評估
- `matplotlib` / `seaborn`：資料視覺化與混淆矩陣繪圖


In [1]:
import re
from pathlib import Path
from pprint import pprint

import jieba
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager as fm
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 777
BASE_DIR = Path.cwd()
DICT_DIR = BASE_DIR.parent / 'dict'
DATA_DIR = BASE_DIR.parent / 'data'

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

設定中文字體

In [2]:
font_path = BASE_DIR / 'font.ttf'
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    font_name = fm.FontProperties(fname=str(font_path)).get_name()
else:
    font_name = 'Arial Unicode MS'

plt.rcParams['font.sans-serif'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False
print(f'using font: {font_name}')

using font: Noto Sans TC


## 2. 文字前處理
本節對應範例，先讀入原始資料並檢查資料狀況，再完成清理、合併文字與斷詞。

- 資料來源：`ptt_iran_20260301_0410_full.csv`

In [3]:
# 讀取原始 PTT 資料，並建立 summarize_raw_dataset() 函式檢查資料筆數、重複網址、缺值與日期範圍。
# 這一格的輸出 raw_summary 可以用來確認資料是否完整、是否有重複文章，以及標籤欄位是否有缺失。

full_raw = pd.read_csv(DATA_DIR / 'ptt_iran_20260301_0410_full.csv')


# 定義一個摘要函式，用來檢查原始資料集的品質
def summarize_raw_dataset(df, dataset_name):
    # 將文章日期轉成 datetime，無法轉換者會變成 NaT
    art_date = pd.to_datetime(df['artDate'], errors='coerce')
    return {
        'dataset': dataset_name,
        'rows': len(df),
        'unique_urls': df['artUrl'].nunique(dropna=True),
        'duplicate_urls': df['artUrl'].duplicated().sum(),
        'missing_label': df['artCatagory'].isna().sum(),
        'missing_title': df['artTitle'].isna().sum(),
        'missing_content': df['artContent'].isna().sum(),
        'date_min': art_date.min(),
        'date_max': art_date.max(),
    }


# 將摘要結果整理成 DataFrame，方便在 notebook 中以表格呈現
raw_summary = pd.DataFrame([
    summarize_raw_dataset(full_raw, 'ptt_iran_20260301_0410_full.csv'),
])

display(raw_summary)


,dataset,rows,unique_urls,duplicate_urls,missing_label,missing_title,missing_content,date_min,date_max
0,ptt_iran_20260301_0410_full.csv,3547,3547,0,0,0,0,2026-03-01 00:01:58,2026-04-10 22:02:53


從上表可以檢查完整資料集的文章數、重複網址、缺值狀況與日期範圍。

In [4]:
# 預覽原始資料前三筆，快速確認欄位名稱與內容格式。

full_raw.head(3)


,system_id,artUrl,artTitle,artDate,artPoster,artCatagory,artContent,artComment,insertedDate,dataSource
0,1,https://www.ptt.cc/bbs/Military/M.1775823165.A.7A5.html,[新聞] 保護伊朗人員前來 巴基斯坦空軍闢安全,2026-04-10 20:12:41,wysptt,Military,新聞出處：\nhttps://www.cna.com.tw/news/aopl/202604100193.aspx\n\n新聞內容摘要：\n（中央社新德里10日綜合外電報導）美國與伊朗將於11日在巴基斯坦首都伊斯蘭馬巴德舉行\n萬眾...,"[{""cmtStatus"": ""推"", ""cmtPoster"": ""tony121010"", ""cmtContent"": "": 那麼多石油王，他們會出啦"", ""cmtDate"": ""2026-04-10 20:13:00""}, {""...",2026-04-13 22:25:19,ptt
1,2,https://www.ptt.cc/bbs/Military/M.1775809255.A.D32.html,[情報] 幾則關於伊朗戰爭的民調,2026-04-10 16:20:52,jimmy5680,Military,https://x.com/NateSilver538/status/2039839743404363863\n根據Nate Silver的統整各間民調，\n可見美國民眾對伊朗戰爭的支持度持續下滑\nhttps://pbs.twim...,"[{""cmtStatus"": ""推"", ""cmtPoster"": ""MarchelKaton"", ""cmtContent"": "": 川普還嗆右派名嘴不是MAGA自己人了"", ""cmtDate"": ""2026-04-10 17:53:...",2026-04-13 22:25:19,ptt
2,3,https://www.ptt.cc/bbs/Military/M.1775807658.A.5CA.html,[分享] 大西洋期刊：中國從伊朗戰爭學到什麼,2026-04-10 15:54:15,jimmy5680,Military,大西洋期刊，我對這篇文章不予置評，僅供參考\nhttps://x.com/shustry/status/2042217021010571332\nhttps://t.co/e88nkwJ8hQ\n\n中國從伊朗戰爭中學到了什麼\n\...,"[{""cmtStatus"": ""→"", ""cmtPoster"": ""jinkela1"", ""cmtContent"": "": 可寫一整篇回文 但總而言之 台灣情況和伊朗天差地別"", ""cmtDate"": ""2026-04-10 15:...",2026-04-13 22:25:20,ptt


In [5]:
# 統計 artCatagory 欄位的類別分布，檢查 Gossiping、Military 以及缺失值各有多少筆。

label_preview = full_raw['artCatagory'].value_counts(dropna=False).to_frame(name='full_raw')

label_preview


,full_raw
artCatagory,
Gossiping,2616
Military,931


### 2.1 清理
會沿用已出現的 PTT 專屬清理邏輯。

處理重點：
- 移除缺少標題、內文或版別的資料
- 移除重複 `artUrl`
- 清掉網址、PTT 發信站資訊、引述、回文標籤與雜訊符號
- 合併 `artTitle` 與 `artContent` 成新的 `content`


In [6]:
# 建立文字前處理流程：自訂詞、同義詞、停用詞、PTT 文字清理、jieba 斷詞、資料整理與混淆矩陣繪圖函式。
# 這一格是後續分類模型的核心前處理設定，會把原始文章轉成模型可以使用的 words 欄位。

# 自訂專有名詞，讓 jieba 不會把重要詞彙切得太碎
custom_words = [
    '防衛隊', '革命衛隊', '無人機', '核設施', '哈瑪斯', '真主黨', '中東',
    '大馬士革', '德黑蘭', '哈梅內伊', '哈米尼', '霍爾木茲海峽',
    '荷姆茲海峽', '荷莫茲海峽'
]

# 同義詞統一表：把不同寫法轉成同一個標準詞
synonyms = {
    '哈梅內伊': '哈米尼',
    '霍爾木茲': '荷姆茲',
    '荷莫茲': '荷姆茲',
    '阿川': '川普',
    '川皇': '川普',
}

# 手動停用詞：移除常見但對分類幫助不大的 PTT 用語與雜訊詞
manual_stopwords = {
    '-----', '----', '連結', '嘖嘖', '醒醒', '裝睡', '衛生紙', '乳摸', '高潮',
    '新聞', '內文', '備註', '刪除', '完整', '轉載', '編輯', '張貼', '署名',
    '文章', '標題', '報導', '八卦', '看板', '作者', '時間', '來源', '內容',
    '媒體', '情況', '問題', '如題', '原因', '網址', '發信', '實業', 'cc',
    'ptt', 'gossiping', 'www', 'com', 'https', 'http', 'share', 'url', 'index',
    'html', 'bbs', 'status', '表示', '行動', '國家', '當作', '出現', '晚間',
    '政府', '目前', '請放', '最後', '違者', '未滿', '繁體', '中文字', '水桶',
    '一天', '以天', '單位', '網友', '點擊', '情報', '分享', '問卦', '討論',
    '心得', '爆卦'
}

# 載入 jieba 字典與使用者自訂詞典
jieba.set_dictionary(str(DICT_DIR / 'dict.txt'))
jieba.load_userdict(str(DICT_DIR / 'user_dict.txt'))
for word in custom_words:
    jieba.add_word(word)

stopwords = []
for stopword_file in [DICT_DIR / 'stopwords.txt', DICT_DIR / 'text_stopwords.txt']:
    with open(stopword_file, encoding='utf-8') as f:
        stopwords.extend([line.strip() for line in f if line.strip()])
stopwords = set(stopwords) | manual_stopwords


# 清理 PTT 原文，只保留適合中文斷詞與建模的主要文字
def clean_ptt_text(text):
    text = str(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'※\s*(發信站|文章網址|編輯).*', '', text)
    text = re.sub(r'引述《.*?》之銘言', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'^[>:]\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'[a-zA-Z0-9]+', '', text)
    text = re.sub(r'[-/=_~]{2,}', '', text)
    text = re.sub(r'[^\u4e00-\u9fa5\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# 將清理後文字斷詞，並移除停用詞與過短詞
def getToken(text):
    tokens = []
    for word in jieba.lcut(text):
        word = synonyms.get(word, word).strip()
        if len(word) >= 2 and word not in stopwords:
            tokens.append(word)
    return tokens


# 將原始 PTT 資料整理成模型可用的文件資料表
def prepare_ptt_documents(df, drop_missing_label=True):
    data = df[['artUrl', 'artTitle', 'artContent', 'artCatagory', 'artDate']].copy()
    subset = ['artTitle', 'artContent'] + (['artCatagory'] if drop_missing_label else [])
    data = data.dropna(subset=subset)
    data = data.drop_duplicates(subset='artUrl').copy()
    data['artDate'] = pd.to_datetime(data['artDate'], errors='coerce')
    data['title_clean'] = data['artTitle'].map(clean_ptt_text)
    data['content_clean'] = data['artContent'].map(clean_ptt_text)
    data['content'] = (data['title_clean'] + ' ' + data['content_clean']).str.strip()
    data['words'] = data['content'].apply(getToken).map(' '.join)
    data = data[data['words'].str.len() > 0].copy()
    return data


# 繪製混淆矩陣，用來觀察模型的正確與錯誤分類情況
def plot_confusion(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, ax=ax)
    ax.set(
        xlabel='Pred',
        ylabel='True',
        xticklabels=classes,
        yticklabels=classes,
        title=title,
    )
    plt.yticks(rotation=0)
    plt.show()
    return cm

Building prefix dict from /Users/wuchunting/Downloads/SMA_2026S-main/Social_Network_Analysis_Group_3_work2-main/groupwork2/dict/dict.txt ...
Loading model from cache /var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/jieba.u3cdd2ac410c7ea92c26c4a2c6e3f75b6.cache
Loading model cost 0.163 seconds.
Prefix dict has been built successfully.


文章的標題 `artTitle` 和內文 `artContent` 都會納入分析內容，並保留 `artUrl` 與 `artCatagory` 供後續建模與比對使用。

In [7]:
# 呼叫 prepare_ptt_documents() 清理完整資料集，並將清理後的 full_docs 複製成 train_docs 供後續模型使用。
# clean_summary 用來檢查清理後剩下多少文章、日期範圍，以及兩個分類的資料量。

full_docs = prepare_ptt_documents(full_raw, drop_missing_label=True)

# 本版 notebook 只使用 0410_full.csv。
# 為了讓後續建模流程維持原本寫法，直接把清理後的 full_docs 作為 train_docs 使用。
train_docs = full_docs.copy()

clean_summary = pd.DataFrame([
    {
        'dataset': 'train_docs_from_0410_full',
        'rows_after_clean': len(train_docs),
        'date_min': train_docs['artDate'].min(),
        'date_max': train_docs['artDate'].max(),
        'Gossiping': (train_docs['artCatagory'] == 'Gossiping').sum(),
        'Military': (train_docs['artCatagory'] == 'Military').sum(),
    }
])

display(clean_summary)


,dataset,rows_after_clean,date_min,date_max,Gossiping,Military
0,train_docs_from_0410_full,3547,2026-03-01 00:01:58,2026-04-10 22:02:53,2616,931


In [8]:
# 預覽清理後的文章資料，確認 content 是清理後全文，words 是斷詞後的模型輸入文字。

train_docs[['artDate', 'artCatagory', 'artTitle', 'content', 'words']].head(3)

,artDate,artCatagory,artTitle,content,words
0,2026-04-10 20:12:41,Military,[新聞] 保護伊朗人員前來 巴基斯坦空軍闢安全,保護伊朗人員前來 巴基斯坦空軍闢安全 新聞出處 新聞內容摘要 中央社新德里日綜合外電報導 美國與伊朗將於日在巴基斯坦首都伊斯蘭馬巴德舉行 萬眾矚目的談判 外媒披露 巴基斯坦空軍出動大批各式軍機到波斯灣空域 保護伊朗代表 團前來赴會 ...,保護 伊朗 人員 前來 巴基斯坦 空軍 安全 出處 摘要 中央社 新德里 綜合 外電 美國 伊朗 將於日 巴基斯坦 首都 伊斯蘭 巴德 舉行 萬眾矚目 談判 外媒 披露 巴基斯坦 空軍 出動 大批 軍機 波斯灣 空域 保護 伊朗 代...
1,2026-04-10 16:20:52,Military,[情報] 幾則關於伊朗戰爭的民調,幾則關於伊朗戰爭的民調 根據 的統整各間民調 可見美國民眾對伊朗戰爭的支持度持續下滑 皮尤民調 追蹤美國民眾對以色列和納坦雅胡的支持度 顯示美國民眾越來越厭惡以色列和納坦雅胡 天空新聞 民調顯示從去年年底到現在 英國民眾認為美國是世...,幾則 伊朗 戰爭 民調 統整 各間 民調 美國 民眾 伊朗 戰爭 支持度 持續 下滑 民調 追蹤 美國 民眾 以色列 納坦雅胡 支持度 顯示 美國 民眾 越來越 厭惡 以色列 納坦雅胡 天空 民調 顯示 去年 年底 現在 英國 民眾...
2,2026-04-10 15:54:15,Military,[分享] 大西洋期刊：中國從伊朗戰爭學到什麼,大西洋期刊 中國從伊朗戰爭學到什麼 大西洋期刊 我對這篇文章不予置評 僅供參考 中國從伊朗戰爭中學到了什麼 對台灣實施封鎖 對全球經濟造成的衝擊 將比伊朗封鎖荷姆茲海峽更為嚴重 中國可以從這場伊朗戰爭中學到許多 美國劃下的紅線與期限...,大西洋 期刊 中國 伊朗 戰爭 學到 大西洋 期刊 這篇 不予置評 參考 中國 伊朗 戰爭 中學 台灣 實施 封鎖 全球 經濟 造成 衝擊 將比 伊朗 封鎖 荷姆茲海峽 嚴重 中國 這場 伊朗 戰爭 中學 美國 劃下 紅線 期限 伴...


### 2.2 斷詞
使用專案字典、使用者字典、停用字與已整理過的同義詞規則。

In [9]:
# 只查看分類標籤與斷詞結果，方便檢查各類文章的關鍵詞是否合理。

train_docs[['artCatagory', 'words']].head(5)

,artCatagory,words
0,Military,保護 伊朗 人員 前來 巴基斯坦 空軍 安全 出處 摘要 中央社 新德里 綜合 外電 美國 伊朗 將於日 巴基斯坦 首都 伊斯蘭 巴德 舉行 萬眾矚目 談判 外媒 披露 巴基斯坦 空軍 出動 大批 軍機 波斯灣 空域 保護 伊朗 代...
1,Military,幾則 伊朗 戰爭 民調 統整 各間 民調 美國 民眾 伊朗 戰爭 支持度 持續 下滑 民調 追蹤 美國 民眾 以色列 納坦雅胡 支持度 顯示 美國 民眾 越來越 厭惡 以色列 納坦雅胡 天空 民調 顯示 去年 年底 現在 英國 民眾...
2,Military,大西洋 期刊 中國 伊朗 戰爭 學到 大西洋 期刊 這篇 不予置評 參考 中國 伊朗 戰爭 中學 台灣 實施 封鎖 全球 經濟 造成 衝擊 將比 伊朗 封鎖 荷姆茲海峽 嚴重 中國 這場 伊朗 戰爭 中學 美國 劃下 紅線 期限 伴...
3,Military,以色列 軍方 認為 伊朗 以色列 以色列 軍方 告訴 國會 議員 伊朗 戰前 以色列 軍方 另稱 作戰 伊朗 造成 嚴重 打擊 伊朗 著手 恢復 飛彈 生產 能力 肖像 掛在 辦公室 掛上 孩子 照片 決定 看看 眼睛
4,Military,曾嗆 發展 核武 伊朗 外長 傷重 不治 揚言 發展 核武 伊朗 外長 拉齊 遇襲 住所 攻擊 罹難 伊朗 伊朗 外交部長 哈拉 美國 以色列 空襲 受傷 今天 傷重 不治 哈拉 德黑蘭 尋求 發展 核武 法新社 哈拉 濟生 隸屬於...


### 2.3 資料集基本檢視
先確認清理後可用於分類的文章數與版別分布。

In [10]:
# 輸出清理後的文章數、日期範圍與類別筆數。
# 長條圖用來視覺化類別分布，確認資料是否有明顯不平衡。

print(f"total posts: {len(train_docs['artUrl'].unique())}")
print(f"date range: {train_docs['artDate'].min()} ~ {train_docs['artDate'].max()}")
print('category count:')
print(train_docs['artCatagory'].value_counts())

fig, ax = plt.subplots(figsize=(5, 4))
train_docs['artCatagory'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Cleaned Full Data Label Distribution')
ax.set_xlabel('artCatagory')
ax.set_ylabel('count')
plt.xticks(rotation=0)
plt.show()


total posts: 3547
date range: 2026-03-01 00:01:58 ~ 2026-04-10 22:02:53
category count:
artCatagory
Gossiping    2616
Military      931
Name: count, dtype: int64


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/4501940.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 分類模型的訓練流程
以 Logistic Regression 分類流程，再比較不同文字表示方式與評估方法。

In [11]:
# 設定模型輸入 X 與答案 y，並使用 train_test_split 切成訓練集與測試集。
# stratify=y 會讓訓練集與測試集維持接近原始資料的類別比例。

data = train_docs.copy()
# X 是輸入特徵：斷詞後的文章文字
X = data['words']
# y 是預測目標：文章所屬看板分類
y = data['artCatagory']

# 將資料分成訓練集與測試集，測試集比例為 30%
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(X_train.head())
print(y_train.head())

2704    伊朗 無人機 烏俄 戰爭 俄國 戰車 打爆 現在 烏克蘭 頭痛 伊朗 無人機 以前 一部 影片 底迪 哈米尼 殉教 哈米尼 回答 八十歲 考慮 哈米尼 只差 講出 真心話 現在 無人機 殉教 成本 極低 台萬 伊朗 以前 軍工 大國...
1603    美國 疑在 伊朗 附近 村莊 投反 戰車 地雷 記者 美國 疑在 伊朗 附近 村莊 投反 戰車 地雷 分析師 開源 研究 組織 週四 指出 美國 伊朗 南部 村莊 投下 戰車 地雷 此前 社群 流傳 影像 似乎 顯示 拉子 南部 郊...
2205    伊朗 根本 鄰居 縱火 瓦斯桶 亂炸 鄰居 美軍 基地 以外 知道 真正 搞事 魷魚 魷魚 滅國 老美 輸血 魷魚 有產 石油 還能 源源不絕 挑事 免費 裝備 伊斯蘭 兄弟 老美 沆瀣一氣 美元 魷魚 輸血 血袋 魷魚 表兄弟 伊...
3449    杜拜 機場 伊朗 自殺 無人機 攻擊 女滿 杜拜 機場 伊朗 自殺 無人機 攻擊 女滿 臉血 逃命 畫面 曝光 聯合 新聞網 聯合報 編譯 盧思綸 即時 杜拜 機場 多名 人員 受傷 擷自 影音 每日 郵報 美國 以色列 伊朗 發動...
1278    伊朗 打完 投資 伊朗 打贏 美帝 直接 世界 強權 掌控 世界 石油 出口 世界 向來 贏幫 伊朗 多數 能力 半信半疑 現在 戰成 以後 世界 搶著 打好 關係 幫忙 重建 伊朗 戰後 值得 投資 沒人敢 制裁 石油 制裁 美帝...
Name: words, dtype: object
2704    Gossiping
1603    Gossiping
2205    Gossiping
3449    Gossiping
1278    Gossiping
Name: artCatagory, dtype: object


In [12]:
# 比較原始資料、訓練集與測試集的類別百分比，檢查切分後是否仍維持類別比例。

print('raw data percentage:')
print((data['artCatagory'].value_counts(normalize=True) * 100).round(2))
print('\ntrain percentage:')
print((y_train.value_counts(normalize=True) * 100).round(2))
print('\ntest percentage:')
print((y_test.value_counts(normalize=True) * 100).round(2))

raw data percentage:
artCatagory
Gossiping    73.75
Military     26.25
Name: proportion, dtype: float64

train percentage:
artCatagory
Gossiping    73.77
Military     26.23
Name: proportion, dtype: float64

test percentage:
artCatagory
Gossiping    73.71
Military     26.29
Name: proportion, dtype: float64


### 3.2 將文章轉為 DTM
DTM（document-term matrix）

- `CountVectorizer`：以詞頻表示文章
- `TfidfVectorizer`：以 TF-IDF 權重表示文章

使用 `CountVectorizer + LogisticRegression` 流程。

In [13]:
# 建立 CountVectorizer，把文章轉成詞頻矩陣；max_features=1000 表示最多保留 1000 個詞作為特徵。

count_vectorizer = CountVectorizer(max_features=1000)
print(count_vectorizer)

CountVectorizer(max_features=1000)


In [14]:
# 用訓練資料 fit CountVectorizer 並轉換成詞頻矩陣。
# 再轉成 DataFrame 預覽前幾篇文章在前 20 個特徵詞上的出現次數。

vec_train_count = count_vectorizer.fit_transform(X_train)
count_vocab = count_vectorizer.get_feature_names_out()
count_df = pd.DataFrame(vec_train_count.toarray(), columns=count_vocab)
count_df.iloc[:5, :20]

,一下,一份,一位,一名,一堆,一場,一座,一枚,一架,一次,一段,一直,一種,一致,一艘,一處,一部分,一項,一點,三日
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
# 用已經 fit 好的 CountVectorizer 轉換測試集，並印出訓練矩陣與測試矩陣的形狀。

vec_test_count = count_vectorizer.transform(X_test)
print(vec_train_count.shape)
print(vec_test_count.shape)

(2482, 1000)
(1065, 1000)


In [16]:
# 建立並訓練 Logistic Regression 模型，使用 CountVectorizer 產生的詞頻特徵做分類。

count_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
count_logistic.fit(vec_train_count, y_train)
count_logistic


LogisticRegression(max_iter=2000, random_state=777)

In [17]:
# 查看模型學到的類別順序；這個順序會影響 predict_proba 機率欄位的對應類別。

count_logistic.classes_

array(['Gossiping', 'Military'], dtype=object)

使用 train set 訓練後，再用 test set 查看模型的分類結果。

In [18]:
# 用 CountVectorizer + Logistic Regression 預測測試集，並取得每筆文章屬於各類別的預測機率。

y_pred_count = count_logistic.predict(vec_test_count)
y_pred_proba_count = count_logistic.predict_proba(vec_test_count)
print(y_pred_count[:10])

['Gossiping' 'Gossiping' 'Military' 'Gossiping' 'Gossiping' 'Gossiping'
 'Military' 'Military' 'Military' 'Gossiping']


觀察看看模型輸出的類別機率。

In [19]:
# 查看預測機率矩陣大小，並顯示第一篇測試文章對每個類別的預測機率。

print(y_pred_proba_count.shape)
pd.Series(y_pred_proba_count[0], index=count_logistic.classes_)

(1065, 2)


Gossiping    0.998059
Military     0.001941
dtype: float64

### 3.3 模型評估
使用 `classification_report` 與混淆矩陣檢查 `CountVectorizer + LogisticRegression` 的表現。

In [20]:
# 輸出 classification_report，觀察 precision、recall、f1-score 與整體 accuracy。

print(classification_report(y_test, y_pred_count))

              precision    recall  f1-score   support

   Gossiping       0.88      0.92      0.90       785
    Military       0.74      0.64      0.68       280

    accuracy                           0.85      1065
   macro avg       0.81      0.78      0.79      1065
weighted avg       0.84      0.85      0.84      1065



混淆矩陣（Confusion Matrix）可以更直接觀察模型在哪一個版別上較容易分錯。

In [21]:
# 繪製 CountVectorizer + Logistic Regression 的混淆矩陣，觀察哪些類別容易被分錯。

classes = list(count_logistic.classes_)
cm_count = plot_confusion(y_test, y_pred_count, classes, 'CountVectorizer + LogisticRegression')
cm_count

/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[722,  63],
       [102, 178]])

### 3.4 TF-IDF
接著改用 `TfidfVectorizer`，比較和純詞頻 DTM 的差異。

In [22]:
# 建立 TF-IDF 向量器，將文字轉成 TF-IDF 權重矩陣。
# TF-IDF 會降低太常見詞的影響，凸顯更能區分類別的詞。

tfidf_vectorizer = TfidfVectorizer(max_features=1000)
vec_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
vec_test_tfidf = tfidf_vectorizer.transform(X_test)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(vec_train_tfidf.toarray(), columns=tfidf_vocab)
tfidf_df.iloc[:5, :20]

,一下,一份,一位,一名,一堆,一場,一座,一枚,一架,一次,一段,一直,一種,一致,一艘,一處,一部分,一項,一點,三日
0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.02538,0.000000,0.0,0.0,0.0,0.029888,0.0,0.024014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.00000,0.088839,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# 使用 TF-IDF 特徵訓練 Logistic Regression，並在測試集上輸出分類表現。

tfidf_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
tfidf_logistic.fit(vec_train_tfidf, y_train)
y_pred_tfidf = tfidf_logistic.predict(vec_test_tfidf)
y_pred_proba_tfidf = tfidf_logistic.predict_proba(vec_test_tfidf)

print(classification_report(y_test, y_pred_tfidf))

              precision    recall  f1-score   support

   Gossiping       0.85      0.97      0.91       785
    Military       0.86      0.53      0.65       280

    accuracy                           0.85      1065
   macro avg       0.86      0.75      0.78      1065
weighted avg       0.85      0.85      0.84      1065



### 3.5 Cross Validation
用 `TfidfVectorizer + LogisticRegression` 檢查 5-fold CV 的整體穩定度。

In [24]:
# 在訓練集內進行 5-fold cross validation，評估 Logistic Regression 的穩定性。
# cv_summary 會整理每個評估指標的平均值與標準差。

cv_logistic = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
cv_train_matrix = TfidfVectorizer(max_features=1000).fit_transform(X_train)

cv_scores = cross_validate(
    cv_logistic,
    cv_train_matrix,
    y_train,
    cv=5,
    scoring=('accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted'),
    return_train_score=False,
)

cv_summary = pd.DataFrame(cv_scores).agg(['mean', 'std']).T
cv_summary

,mean,std
fit_time,0.003382,0.000655
score_time,0.004115,0.000155
test_accuracy,0.837621,0.028023
test_f1_weighted,0.819211,0.034566
test_precision_weighted,0.838541,0.036190
test_recall_weighted,0.837621,0.028023


`cross_val_predict()` 會回傳每一筆訓練資料在 cross-validation 下得到的預測類別。

In [25]:
# 使用 cross_val_predict 取得每筆訓練資料的交叉驗證預測結果，並輸出分類報告。

y_pred_cv = cross_val_predict(cv_logistic, cv_train_matrix, y_train, cv=5)
print(classification_report(y_train, y_pred_cv))

              precision    recall  f1-score   support

   Gossiping       0.84      0.97      0.90      1831
    Military       0.85      0.46      0.60       651

    accuracy                           0.84      2482
   macro avg       0.84      0.72      0.75      2482
weighted avg       0.84      0.84      0.82      2482



## 4. 比較不同模型效果
這一節對應範例 week7 的第 4 節，固定使用 `TfidfVectorizer(max_features=1000)`，比較 Logistic Regression、Decision Tree、SVM 與 Random Forest。

In [26]:
# 定義 train_cv_experiment()：把向量化、交叉驗證、分類報告、混淆矩陣與最終模型訓練包成同一個流程。
# 這個函式後面會被用來比較不同分類模型。

# 定義通用實驗函式，讓不同模型可以用同一套流程比較
def train_cv_experiment(vectorizer, clf, X, y, model_name, cv=5):
    # 將文字轉換成數值矩陣，作為機器學習模型的輸入
    vec_X = vectorizer.fit_transform(X).toarray()
    # 使用交叉驗證取得每筆資料的 out-of-fold 預測結果
    y_pred_cv = cross_val_predict(clf, vec_X, y, cv=cv)
    cls_report = classification_report(y, y_pred_cv, output_dict=True)
    print(f'===== {model_name} =====')
    print(classification_report(y, y_pred_cv))
    plot_confusion(y, y_pred_cv, sorted(pd.Series(y).unique()), f'{model_name} CV Confusion Matrix')
    # 用完整訓練資料重新訓練模型，方便後續拿去預測測試集
    clf.fit(vec_X, y)
    return {
        'report': cls_report,
        'model': clf,
        'vectorizer': vectorizer,
        'cv_pred': y_pred_cv,
    }

In [27]:
# 建立四種模型並逐一進行 TF-IDF + cross validation 實驗，比較 Logistic Regression、Decision Tree、SVM、Random Forest。

model_set = {
    'clf_logistic': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'clf_dtree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'clf_svm': svm.SVC(probability=True, random_state=RANDOM_STATE),
    'clf_rf': RandomForestClassifier(random_state=RANDOM_STATE),
}

result_set = {}
for model_name, model in model_set.items():
    result_set[model_name] = train_cv_experiment(
        TfidfVectorizer(max_features=1000),
        model,
        X_train,
        y_train,
        model_name,
        cv=5,
    )

===== clf_logistic =====
              precision    recall  f1-score   support

   Gossiping       0.84      0.97      0.90      1831
    Military       0.85      0.46      0.60       651

    accuracy                           0.84      2482
   macro avg       0.84      0.72      0.75      2482
weighted avg       0.84      0.84      0.82      2482



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_dtree =====
              precision    recall  f1-score   support

   Gossiping       0.84      0.85      0.85      1831
    Military       0.56      0.55      0.56       651

    accuracy                           0.77      2482
   macro avg       0.70      0.70      0.70      2482
weighted avg       0.77      0.77      0.77      2482



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_svm =====
              precision    recall  f1-score   support

   Gossiping       0.84      0.97      0.90      1831
    Military       0.84      0.49      0.62       651

    accuracy                           0.84      2482
   macro avg       0.84      0.73      0.76      2482
weighted avg       0.84      0.84      0.83      2482



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


===== clf_rf =====
              precision    recall  f1-score   support

   Gossiping       0.84      0.98      0.90      1831
    Military       0.88      0.47      0.61       651

    accuracy                           0.84      2482
   macro avg       0.86      0.72      0.76      2482
weighted avg       0.85      0.84      0.83      2482



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


分別觀察各個分類模型在 weighted avg 指標上的整體表現。

In [28]:
# 把各模型的 cross validation 結果整理成表格，並依 weighted f1-score 由高到低排序。

model_summary = pd.DataFrame([
    {
        'model': model_name,
        'accuracy': result['report']['accuracy'],
        'precision_weighted': result['report']['weighted avg']['precision'],
        'recall_weighted': result['report']['weighted avg']['recall'],
        'f1_weighted': result['report']['weighted avg']['f1-score'],
        'f1_macro': result['report']['macro avg']['f1-score'],
    }
    for model_name, result in result_set.items()
]).sort_values(by='f1_weighted', ascending=False)

model_summary

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro
2,clf_svm,0.842869,0.842596,0.842869,0.827863,0.761794
3,clf_rf,0.844077,0.849770,0.844077,0.826053,0.756775
0,clf_logistic,0.837631,0.839397,0.837631,0.819729,0.748606
1,clf_dtree,0.770749,0.769294,0.770749,0.769999,0.701871


找出 `weighted avg f1-score` 表現最好的模型。

In [29]:
# 取出表現最好的模型名稱與完整結果，並列印最佳模型的分類報告字典。

best_model_name = model_summary.iloc[0]['model']
best_result = result_set[best_model_name]

print(f'best model: {best_model_name}')
pprint(best_result['report'])

best model: clf_svm
{'Gossiping': {'f1-score': 0.9007633587786259,
               'precision': 0.8432586946164841,
               'recall': 0.9666848716548334,
               'support': 1831.0},
 'Military': {'f1-score': 0.6228239845261122,
              'precision': 0.8407310704960835,
              'recall': 0.4946236559139785,
              'support': 651.0},
 'accuracy': 0.8428686543110395,
 'macro avg': {'f1-score': 0.761793671652369,
               'precision': 0.8419948825562837,
               'recall': 0.730654263784406,
               'support': 2482.0},
 'weighted avg': {'f1-score': 0.8278630635979706,
                  'precision': 0.8425957279354281,
                  'recall': 0.8428686543110395,
                  'support': 2482.0}}


In [30]:
# 使用最佳模型預測保留的測試集，輸出 classification_report 並繪製測試集混淆矩陣。

best_model = best_result['model']
best_vectorizer = best_result['vectorizer']

y_pred_best_test = best_model.predict(best_vectorizer.transform(X_test).toarray())
print(classification_report(y_test, y_pred_best_test))
plot_confusion(y_test, y_pred_best_test, sorted(y_test.unique()), f'{best_model_name} on Held-out Test Set')

              precision    recall  f1-score   support

   Gossiping       0.86      0.98      0.91       785
    Military       0.90      0.55      0.68       280

    accuracy                           0.86      1065
   macro avg       0.88      0.76      0.80      1065
weighted avg       0.87      0.86      0.85      1065



/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[767,  18],
       [126, 154]])

## 5. 分析可解釋模型的結果
雖然的最佳模型不一定是 Logistic Regression，但 Logistic Regression 的係數最容易解釋，因此仍沿用範例 notebook 的做法，用 `TfidfVectorizer + LogisticRegression` 觀察字詞特徵。

In [31]:
# 定義 plot_coef()，用 Logistic Regression 的係數找出最能推動各類別判斷的關鍵詞。
# 係數越大代表該詞越會把文章推向該類別；係數越小則代表越不支持該類別。

# 定義係數視覺化函式，找出 Logistic Regression 最重視的關鍵詞
def plot_coef(logistic_reg_model, feature_names, top_n=10):
    # 二元分類時 sklearn 只會存一組係數，另一類可視為相反方向
    if len(logistic_reg_model.classes_) == 2:
        coef_df = pd.DataFrame(
            {
                logistic_reg_model.classes_[0]: -logistic_reg_model.coef_[0],
                logistic_reg_model.classes_[1]: logistic_reg_model.coef_[0],
            },
            index=feature_names,
        )
    else:
        coef_df = pd.DataFrame(
            logistic_reg_model.coef_.T,
            columns=logistic_reg_model.classes_,
            index=feature_names,
        )

    display(coef_df.head())

    for label in coef_df.columns:
        select_words = (
            coef_df[[label]]
            .sort_values(by=label, ascending=False)
            .iloc[np.r_[0:top_n, -top_n:0]]
        )
        # 正係數與負係數使用不同顏色，方便區分詞彙對分類的影響方向
        colors = np.where(select_words[label] >= 0, 'darkseagreen', 'rosybrown')

        fig, ax = plt.subplots(figsize=(8, top_n * 0.8))
        ax.barh(select_words.index, select_words[label], color=colors)
        ax.invert_yaxis()
        ax.set_title(f'Coeff increase/decrease odds ratio of 「{label}」 label the most', loc='left', size=14)
        ax.set_ylabel('word', size=12)
        ax.set_xlabel('odds ratio', size=12)
        plt.show()

In [32]:
# 呼叫 plot_coef()，畫出 TF-IDF + Logistic Regression 每個類別最重要的正向與負向關鍵詞。

plot_coef(tfidf_logistic, tfidf_vectorizer.get_feature_names_out(), top_n=10)


,Gossiping,Military
一下,0.931109,-0.931109
一份,-0.211389,0.211389
一位,-0.701111,0.701111
一名,-0.591839,0.591839
一堆,0.289437,-0.289437


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/4078216469.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/4078216469.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


從係數圖可以看到，模型確實學到兩個版別不同的語言風格。

- `Military` 端通常更容易出現軍事情勢、地緣政治與戰況相關詞，例如空襲、伊拉克、官員、荷姆茲海峽等。
- `Gossiping` 端則更常出現 PTT 問卦或社會討論口吻，例如台灣、有沒有、現在、美國等。
- 這也說明本次分類不只是辨識「伊朗」主題，而是同時辨識兩個版在敘事方式與詞彙選擇上的差異。


## 6. 使用測試集分析最佳模型的錯誤分類結果
原本範例會把分類器套用到另一份資料；但本版 notebook 只使用 `ptt_iran_20260301_0410_full.csv` 一份完整資料，因此這裡改成分析第 3 節切出的 **held-out test set**。

這樣可以避免再使用 `ptt_iran_20260301_0320_new.csv`，同時仍能檢查最佳模型在未參與訓練資料上的實際表現。


In [33]:
# 建立 test_result，把測試集文章資訊與最佳模型預測結果放在同一張表中，方便後續錯誤分析。

test_result = data.loc[X_test.index, ['artDate', 'artCatagory', 'artTitle', 'words']].copy()
test_result['pred'] = y_pred_best_test

test_summary = pd.DataFrame({
    'rows': [len(test_result)],
    'date_min': [test_result['artDate'].min()],
    'date_max': [test_result['artDate'].max()],
    'Gossiping': [(test_result['artCatagory'] == 'Gossiping').sum()],
    'Military': [(test_result['artCatagory'] == 'Military').sum()],
})

test_summary


,rows,date_min,date_max,Gossiping,Military
0,1065,2026-03-01 00:01:58,2026-04-10 22:02:53,785,280


從清理後的資料中，以 7:3 切分出來的測試集。它沒有被最佳模型拿來訓練，因此可以用來觀察模型在未見資料上的分類效果。


In [34]:
# 再次輸出最佳模型在測試集上的分類報告，並預覽前 10 筆測試結果。

print(classification_report(test_result['artCatagory'], test_result['pred']))

test_result[['artDate', 'artCatagory', 'pred', 'artTitle', 'words']].head(10)


              precision    recall  f1-score   support

   Gossiping       0.86      0.98      0.91       785
    Military       0.90      0.55      0.68       280

    accuracy                           0.86      1065
   macro avg       0.88      0.76      0.80      1065
weighted avg       0.87      0.86      0.85      1065



,artDate,artCatagory,pred,artTitle,words
2391,2026-03-12 21:32:14,Gossiping,Gossiping,[爆卦] 伊朗最高領袖命令美國所有中東基地關閉,伊朗 領袖 命令 美國 中東 基地 關閉 伊朗 領袖 就任 首次 發表 演說 伊朗 新任 領袖 一條 重要 石油 運輸 路線 繼續 關閉 伊朗 新任 領袖 穆傑 塔巴 哈米尼 發表 講話 呼籲 伊朗 團結一致 警告 施壓 工具 重要...
1568,2026-03-29 13:09:26,Gossiping,Gossiping,[問卦] 伊朗為何不肯加入西方陣營,伊朗 加入 西方 陣營 國力 夠強 兩邊 想要 好過 通常 加入 西方 陣營 就連 中國 阿富汗 戰爭 脫離 蘇聯 美國 合作 經濟 改革 開放 今天 幾年 關係 緊張 伊朗 明明 石油 要過 日子 不難 西方 作對 看看 台日 還...
3291,2026-03-02 07:38:59,Gossiping,Gossiping,Re: [新聞] 川普：伊朗新任領導層想談判「我同意了」,川普 伊朗 新任 領導層 談判 同意 提到 過往 參與 相關 談判 伊朗 外交 人員 去世 早點 本來 達成 協議 謹慎 了解 阿富汗 伊朗 這種 教義派 穆斯林 狀態 實質 意義 中央 指揮權 哈米尼 這種 共主 名義 共主 真正...
2776,2026-03-06 08:21:43,Gossiping,Gossiping,Re: [爆卦] 美國表示 伊朗戰爭可能持續到9月,美國 伊朗 戰爭 持續 當初 軍武 版講 伊朗 戰爭 打久 台灣 不利 美國 以色列 拖入 派出 地上 部隊 美國人 反對 伊朗 戰爭 正常 軍武 一堆 呆子 還在 嘲諷 美國人 孤立 主義 講個 台灣人 愛聽 戰爭 持續 下去 中...
1280,2026-04-03 23:34:23,Gossiping,Gossiping,[爆卦] 被擊落美軍飛行員之一已獲救撤離伊朗,擊落 美軍 飛行員 獲救 撤離 伊朗 以色列 消息 擊落 攻擊 戰鬥機 兩名 美國 機組 人員 獲救 並從 南部 撤離 美國 官員 證實 一名 飛行員 撤離
3241,2026-03-02 12:38:28,Gossiping,Gossiping,[問卦] 為什麼還不出兵攻佔伊朗？,還不 出兵 攻佔 伊朗 美國 爸爸 美國 爸爸 爸爸 伊朗 身為 孫子 島國 攻佔 伊朗 反共 基地 幾桶 石油 回家
2687,2026-03-07 13:55:36,Gossiping,Gossiping,[問卦] 庫德族：敢死隊願加入推翻伊朗政權,庫德族 敢死隊 加入 推翻 伊朗 政權 庫德族 表態 願意 加入 推翻 伊朗 政權 川普 表態 戰到 伊朗 無條件 投降 無力 反抗 派出 美軍 地面 部隊 後續 發展
2629,2026-03-08 20:17:24,Gossiping,Gossiping,[新聞] 中東大戰第9天！伊朗「4波飛彈」狂轟以,中東 大戰 第天 伊朗 飛彈 狂轟 記者 記者 閔文昱 綜合 中東 大戰 第天 伊朗 飛彈 狂轟 以色列 全境 警報 大作 美國 以色列 伊朗 發動 軍事 進入 第天 中東 局勢 持續 升溫 以色列 晨全境 響起 防空 警報 軍方 ...
241,2026-03-30 23:19:52,Military,Military,[分享] 川普與盧比歐威脅伊朗必須開放荷姆茲海峽,川普 盧比歐 威脅 伊朗 開放 荷姆茲海峽 美利堅 合眾國 理性 伊朗 政權 進行 嚴肅 談判 結束 伊朗 談判 取得 重大 進展 若因 雙方 未能 短期 達成 協議 認為 達成 荷姆茲海峽 未能 立即 恢復 通行 將以 摧毀 徹底...
2826,2026-03-05 17:01:05,Gossiping,Gossiping,[問卦] 北韓：支援伊朗飛彈 一發就把以色列幹掉,北韓 支援 伊朗 飛彈 一發 以色列 幹掉 金正恩 發表 措辭 強硬 聲明 表達 德黑蘭 支持 伊朗 提出 要求 北韓 準備 伊朗 提供 飛彈 用於 打擊 以色列 金正恩 一枚 導彈 足以 消滅 色列 言論 引起 國際 社會 關注 ...


In [35]:
# 繪製最佳模型在測試集上的混淆矩陣，檢查真實類別與預測類別的對應情況。

plot_confusion(
    test_result['artCatagory'],
    test_result['pred'],
    sorted(test_result['artCatagory'].unique()),
    f'{best_model_name} on Held-out Test Set',
)


/var/folders/dn/crq7pws16_zdbgt0gftcq5g00000gn/T/ipykernel_3006/3735374400.py:99: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[767,  18],
       [126, 154]])

In [36]:
# 再次顯示測試集前 10 筆資料，方便與下一步錯誤分類分析銜接。

test_result[['artDate', 'artCatagory', 'pred', 'artTitle', 'words']].head(10)


,artDate,artCatagory,pred,artTitle,words
2391,2026-03-12 21:32:14,Gossiping,Gossiping,[爆卦] 伊朗最高領袖命令美國所有中東基地關閉,伊朗 領袖 命令 美國 中東 基地 關閉 伊朗 領袖 就任 首次 發表 演說 伊朗 新任 領袖 一條 重要 石油 運輸 路線 繼續 關閉 伊朗 新任 領袖 穆傑 塔巴 哈米尼 發表 講話 呼籲 伊朗 團結一致 警告 施壓 工具 重要...
1568,2026-03-29 13:09:26,Gossiping,Gossiping,[問卦] 伊朗為何不肯加入西方陣營,伊朗 加入 西方 陣營 國力 夠強 兩邊 想要 好過 通常 加入 西方 陣營 就連 中國 阿富汗 戰爭 脫離 蘇聯 美國 合作 經濟 改革 開放 今天 幾年 關係 緊張 伊朗 明明 石油 要過 日子 不難 西方 作對 看看 台日 還...
3291,2026-03-02 07:38:59,Gossiping,Gossiping,Re: [新聞] 川普：伊朗新任領導層想談判「我同意了」,川普 伊朗 新任 領導層 談判 同意 提到 過往 參與 相關 談判 伊朗 外交 人員 去世 早點 本來 達成 協議 謹慎 了解 阿富汗 伊朗 這種 教義派 穆斯林 狀態 實質 意義 中央 指揮權 哈米尼 這種 共主 名義 共主 真正...
2776,2026-03-06 08:21:43,Gossiping,Gossiping,Re: [爆卦] 美國表示 伊朗戰爭可能持續到9月,美國 伊朗 戰爭 持續 當初 軍武 版講 伊朗 戰爭 打久 台灣 不利 美國 以色列 拖入 派出 地上 部隊 美國人 反對 伊朗 戰爭 正常 軍武 一堆 呆子 還在 嘲諷 美國人 孤立 主義 講個 台灣人 愛聽 戰爭 持續 下去 中...
1280,2026-04-03 23:34:23,Gossiping,Gossiping,[爆卦] 被擊落美軍飛行員之一已獲救撤離伊朗,擊落 美軍 飛行員 獲救 撤離 伊朗 以色列 消息 擊落 攻擊 戰鬥機 兩名 美國 機組 人員 獲救 並從 南部 撤離 美國 官員 證實 一名 飛行員 撤離
3241,2026-03-02 12:38:28,Gossiping,Gossiping,[問卦] 為什麼還不出兵攻佔伊朗？,還不 出兵 攻佔 伊朗 美國 爸爸 美國 爸爸 爸爸 伊朗 身為 孫子 島國 攻佔 伊朗 反共 基地 幾桶 石油 回家
2687,2026-03-07 13:55:36,Gossiping,Gossiping,[問卦] 庫德族：敢死隊願加入推翻伊朗政權,庫德族 敢死隊 加入 推翻 伊朗 政權 庫德族 表態 願意 加入 推翻 伊朗 政權 川普 表態 戰到 伊朗 無條件 投降 無力 反抗 派出 美軍 地面 部隊 後續 發展
2629,2026-03-08 20:17:24,Gossiping,Gossiping,[新聞] 中東大戰第9天！伊朗「4波飛彈」狂轟以,中東 大戰 第天 伊朗 飛彈 狂轟 記者 記者 閔文昱 綜合 中東 大戰 第天 伊朗 飛彈 狂轟 以色列 全境 警報 大作 美國 以色列 伊朗 發動 軍事 進入 第天 中東 局勢 持續 升溫 以色列 晨全境 響起 防空 警報 軍方 ...
241,2026-03-30 23:19:52,Military,Military,[分享] 川普與盧比歐威脅伊朗必須開放荷姆茲海峽,川普 盧比歐 威脅 伊朗 開放 荷姆茲海峽 美利堅 合眾國 理性 伊朗 政權 進行 嚴肅 談判 結束 伊朗 談判 取得 重大 進展 若因 雙方 未能 短期 達成 協議 認為 達成 荷姆茲海峽 未能 立即 恢復 通行 將以 摧毀 徹底...
2826,2026-03-05 17:01:05,Gossiping,Gossiping,[問卦] 北韓：支援伊朗飛彈 一發就把以色列幹掉,北韓 支援 伊朗 飛彈 一發 以色列 幹掉 金正恩 發表 措辭 強硬 聲明 表達 德黑蘭 支持 伊朗 提出 要求 北韓 準備 伊朗 提供 飛彈 用於 打擊 以色列 金正恩 一枚 導彈 足以 消滅 色列 言論 引起 國際 社會 關注 ...


把錯誤分類的結果篩選出來，觀察哪些文章最容易混淆。


In [37]:
# 篩選出真實分類與預測分類不同的文章，列出模型判斷錯誤的案例。

false_pred = test_result.query('artCatagory != pred').loc[
    :, ['artDate', 'artCatagory', 'pred', 'artTitle', 'words']
]

print(f'錯誤預測筆數: {len(false_pred)} / {len(test_result)}')
false_pred.head(20)


錯誤預測筆數: 144 / 1065


,artDate,artCatagory,pred,artTitle,words
875,2026-03-02 00:38:58,Military,Gossiping,[討論] 伊朗出動殘存的F-4D/E和F-5戰機發起反擊,伊朗 出動 殘存 戰機 發起 反擊 以色列 軍方 影像 伊朗 一架 一架 地面 炸毀 一架 伊朗 準備 起飛 伊朗 剩下 波斯貓 無法 只能 結構 簡單 飛機 進行 反擊 飛機 聯軍 根本 活靶 伊朗 對岸 阿拉伯 輕鬆 火雞 根本...
601,2026-03-09 11:50:27,Military,Gossiping,[新聞] 美以擬派特種部隊 奪伊朗高濃縮鈾,美以 擬派 特種 部隊 伊朗 濃縮鈾 美以 擬派 特種 部隊 伊朗 濃縮鈾 伊朗 公斤 純度 幾週 轉為 武器級 全數 庫存 純度 核彈 阻止 伊朗 取得 核子 武器 美國 總統 川普 公開 宣示 戰爭 目標 伊朗 擁有 四五 公斤...
348,2026-03-23 20:55:46,Military,Gossiping,[新聞] 韓伊外長通話 韓國要求伊朗承諾安全,韓伊 外長 通話 韓國 要求 伊朗 承諾 安全 記者 金泰旭 韓國 外交部 今日 召開 記者會 證實 外交部長 趙顯 伊朗 外交部長 阿巴斯 阿拉 格齊 通話 趙顯 通話 中向 伊朗 表達 韓國 中東 局勢 全球 安全 經濟 不利 ...
39,2026-04-08 14:30:15,Military,Gossiping,Re: [情報] 川普對伊朗談判相關消息數則,川普 伊朗 談判 相關 消息 數則 川普 發文 痛批 伊朗 高層 公告 不實 重點 十點 等於 川普 伊朗 下跪 吉米 先前 補述 似乎 有人還 整理 謠傳 十點 引述 吉米 援用 誤報 打印 字樣 闢謠 截圖 比例 跑掉 圖所示 ...
320,2026-03-25 15:15:01,Military,Gossiping,[討論] 現在其實伊朗就是軍政府治國吧,現在 伊朗 軍政府 治國 談判 事情 軍方 外交部 作用 伊朗 神權 二哈 活著 理論 輪不到 軍方 講話 囚禁 架空 現在 軍方 想談 革命衛隊 不想 革命衛隊 利益 制裁 尋租 盡力 這場 談判 國內 談判 軍政府 同意 二哈要...
507,2026-03-13 14:24:21,Military,Gossiping,[情報] 伊朗持續空襲土耳其,伊朗 持續 空襲 土耳其 伊朗 今天 土耳其 空軍 基地 發射 飛彈 土耳其 之外 北約 盟國 部隊 駐紮 這一 禮拜 伊朗 第三次 空襲 土耳其 先前 土耳其 要來 北約 盟國 額外 一套 愛國 協防 知道 後續 會不會 更多 反應
1164,2026-04-05 14:31:05,Gossiping,Military,[新聞] 專家質疑美國對伊朗Lamerd體育館致命空襲,專家 質疑 美國 伊朗 體育館 致命 空襲 記者 專家 質疑 美國 伊朗 梅爾 體育館 致命 空襲 說法 多名 武器 專家 反駁 美國 說法 美國 聲稱 伊朗 開戰 首日 發生 梅爾 德鎮 一場 致命 空襲 負責 六名 專家 檢視 ...
722,2026-03-05 13:18:05,Military,Gossiping,[情報] 美軍準備對伊朗作戰很可能到今年九月,美軍 準備 伊朗 作戰 今年 九月 節錄 美國 中央 司令部 要求 五角大廈 位於 佛羅里達州 坦帕 總部 增派 更多 軍事 情資 官員 支援 伊朗 至少 為天 持續 這項 消息 政客 取得 一份 通報 文件 川普 首次 伊朗 戰事...
74,2026-04-07 18:23:34,Military,Gossiping,[新聞] 毀滅伊朗倒數計時！航班追蹤器驚見美軍「,毀滅 伊朗 倒數 計時 航班 追蹤器 驚見 美軍 毀滅 伊朗 倒數 計時 航班 追蹤器 驚見 美軍 末日 飛機 起飛 美伊 戰事 膠著 多日 先前 美國 總統 川普 放話 伊朗 期限 達成 和平 協議 該國 恐在 夜間 毀滅 期限 ...
1646,2026-03-27 11:12:30,Gossiping,Military,[新聞] 川普：應伊朗要求 延長停攻10天 談判「進,川普 伊朗 要求 延長 停攻 談判 工商時報 記者 呂佳恩 川普 伊朗 要求 延長 停攻 談判 進展 順利 美國 總統 川普 伊朗 要求 伊朗 能源 設施 暫停 措施 將再 延長 此舉 可望 避免 衝突 進一步 升高 並為 談判 爭...


In [38]:
# 查看真實類別為 Military 但被模型預測錯誤的文章，分析 Military 類文章可能被誤判的原因。

false_pred.loc[false_pred['artCatagory'] == 'Military', :].head(10)


,artDate,artCatagory,pred,artTitle,words
875,2026-03-02 00:38:58,Military,Gossiping,[討論] 伊朗出動殘存的F-4D/E和F-5戰機發起反擊,伊朗 出動 殘存 戰機 發起 反擊 以色列 軍方 影像 伊朗 一架 一架 地面 炸毀 一架 伊朗 準備 起飛 伊朗 剩下 波斯貓 無法 只能 結構 簡單 飛機 進行 反擊 飛機 聯軍 根本 活靶 伊朗 對岸 阿拉伯 輕鬆 火雞 根本...
601,2026-03-09 11:50:27,Military,Gossiping,[新聞] 美以擬派特種部隊 奪伊朗高濃縮鈾,美以 擬派 特種 部隊 伊朗 濃縮鈾 美以 擬派 特種 部隊 伊朗 濃縮鈾 伊朗 公斤 純度 幾週 轉為 武器級 全數 庫存 純度 核彈 阻止 伊朗 取得 核子 武器 美國 總統 川普 公開 宣示 戰爭 目標 伊朗 擁有 四五 公斤...
348,2026-03-23 20:55:46,Military,Gossiping,[新聞] 韓伊外長通話 韓國要求伊朗承諾安全,韓伊 外長 通話 韓國 要求 伊朗 承諾 安全 記者 金泰旭 韓國 外交部 今日 召開 記者會 證實 外交部長 趙顯 伊朗 外交部長 阿巴斯 阿拉 格齊 通話 趙顯 通話 中向 伊朗 表達 韓國 中東 局勢 全球 安全 經濟 不利 ...
39,2026-04-08 14:30:15,Military,Gossiping,Re: [情報] 川普對伊朗談判相關消息數則,川普 伊朗 談判 相關 消息 數則 川普 發文 痛批 伊朗 高層 公告 不實 重點 十點 等於 川普 伊朗 下跪 吉米 先前 補述 似乎 有人還 整理 謠傳 十點 引述 吉米 援用 誤報 打印 字樣 闢謠 截圖 比例 跑掉 圖所示 ...
320,2026-03-25 15:15:01,Military,Gossiping,[討論] 現在其實伊朗就是軍政府治國吧,現在 伊朗 軍政府 治國 談判 事情 軍方 外交部 作用 伊朗 神權 二哈 活著 理論 輪不到 軍方 講話 囚禁 架空 現在 軍方 想談 革命衛隊 不想 革命衛隊 利益 制裁 尋租 盡力 這場 談判 國內 談判 軍政府 同意 二哈要...
507,2026-03-13 14:24:21,Military,Gossiping,[情報] 伊朗持續空襲土耳其,伊朗 持續 空襲 土耳其 伊朗 今天 土耳其 空軍 基地 發射 飛彈 土耳其 之外 北約 盟國 部隊 駐紮 這一 禮拜 伊朗 第三次 空襲 土耳其 先前 土耳其 要來 北約 盟國 額外 一套 愛國 協防 知道 後續 會不會 更多 反應
722,2026-03-05 13:18:05,Military,Gossiping,[情報] 美軍準備對伊朗作戰很可能到今年九月,美軍 準備 伊朗 作戰 今年 九月 節錄 美國 中央 司令部 要求 五角大廈 位於 佛羅里達州 坦帕 總部 增派 更多 軍事 情資 官員 支援 伊朗 至少 為天 持續 這項 消息 政客 取得 一份 通報 文件 川普 首次 伊朗 戰事...
74,2026-04-07 18:23:34,Military,Gossiping,[新聞] 毀滅伊朗倒數計時！航班追蹤器驚見美軍「,毀滅 伊朗 倒數 計時 航班 追蹤器 驚見 美軍 毀滅 伊朗 倒數 計時 航班 追蹤器 驚見 美軍 末日 飛機 起飛 美伊 戰事 膠著 多日 先前 美國 總統 川普 放話 伊朗 期限 達成 和平 協議 該國 恐在 夜間 毀滅 期限 ...
433,2026-03-17 23:27:18,Military,Gossiping,[新聞] 無法昧良心支持川普打伊朗 美國國家反恐,無法 昧良心 支持 川普 伊朗 美國 反恐 原文 聯合報 編譯 高詣 即時 原文 摘要 美國 反恐 中心 主任 肯特 社群 宣布 請職 無法 昧著良心 支持 正在 伊朗 戰爭 肯特 社群 寫給 川普 辭呈 伊朗 美國 並未 構成 迫...
825,2026-03-03 00:41:12,Military,Gossiping,[情報] 卡達擊落兩架伊朗蘇愷24,卡達 擊落 兩架 伊朗 蘇愷 卡達 國防部 卡達 國防部 宣布 偉大 真主 阿拉 之下 擊落 兩架 來犯 伊朗 蘇愷 攔截 七枚 彈道 飛彈 攔截 五架 無人機 阿拉伯 油王們 生氣


In [39]:
# 查看真實類別為 Gossiping 但被模型預測錯誤的文章，分析 Gossiping 類文章可能被誤判的原因。

false_pred.loc[false_pred['artCatagory'] == 'Gossiping', :].head(10)


,artDate,artCatagory,pred,artTitle,words
1164,2026-04-05 14:31:05,Gossiping,Military,[新聞] 專家質疑美國對伊朗Lamerd體育館致命空襲,專家 質疑 美國 伊朗 體育館 致命 空襲 記者 專家 質疑 美國 伊朗 梅爾 體育館 致命 空襲 說法 多名 武器 專家 反駁 美國 說法 美國 聲稱 伊朗 開戰 首日 發生 梅爾 德鎮 一場 致命 空襲 負責 六名 專家 檢視 ...
1646,2026-03-27 11:12:30,Gossiping,Military,[新聞] 川普：應伊朗要求 延長停攻10天 談判「進,川普 伊朗 要求 延長 停攻 談判 工商時報 記者 呂佳恩 川普 伊朗 要求 延長 停攻 談判 進展 順利 美國 總統 川普 伊朗 要求 伊朗 能源 設施 暫停 措施 將再 延長 此舉 可望 避免 衝突 進一步 升高 並為 談判 爭...
1526,2026-03-30 11:07:42,Gossiping,Military,[新聞] 川普：伊朗「已接受大部分」15項停火條,川普 伊朗 接受 大部分 停火 鉅亨網 記者 林羿 川普 伊朗 接受 大部分 停火 條件 美國 總統 川普 德黑蘭 當局 提出 結束 戰爭 目標 要求 伊朗 接受 大部分 條件 外界 無法 確認 雙方 進入 談判 階段 川普 週日 ...
3167,2026-03-02 19:24:02,Gossiping,Military,[新聞] 伊朗發射最先進的飛彈 攻擊以色列總理內,伊朗 發射 先進 飛彈 攻擊 以色列 總理 伊朗 發射 先進 飛彈 攻擊 以色列 總理 唐亞 辦公室 聯合報 編譯 茅毅 即時 伊朗 精銳 伊斯蘭 革命衛隊 聲明 發射 蘭沙 飛彈 攻擊 以色列 總理 唐亞 辦公室 蘭沙 伊朗 先進...
2458,2026-03-11 21:08:48,Gossiping,Military,[新聞] 海格塞斯裁撤了本應調查伊朗學校空襲的辦,海格 塞斯 裁撤 本應 調查 伊朗 學校 空襲 記者 海格 塞斯 裁撤 本應 調查 伊朗 學校 空襲 事件 辦公室 副標題 五角大廈 專注 減緩 平民 傷亡 員工 人數 人驟 降至 不到 國防部長 彼得 海格 塞斯 大幅 裁撤 五角...
2509,2026-03-10 18:49:05,Gossiping,Military,[問卦] 伊朗真的有擊沉油輪嗎???,伊朗 擊沉 油輪 一大堆 不敢過 海峽 伊朗 擊沉 這天 伊朗 擊沉 油輪 波斯灣 變得
1173,2026-04-05 12:58:59,Gossiping,Military,[新聞] 伊朗革命衛隊：擊落MQ-9死神無人機 美軍,伊朗 革命衛隊 擊落 死神 無人機 美軍 劉哲琪 伊朗 革命衛隊 擊落 死神 無人機 美軍 損失 至少 美軍 死神 無人機 示意圖 翻攝 伊朗 伊斯蘭 革命衛隊 稍早 伊斯 法罕 上空 擊落 一架 死神 人機 提供 細節 據報 美國...
2855,2026-03-05 09:43:29,Gossiping,Military,[新聞] 中東局勢再升溫？庫德族傳越境攻打伊朗,中東 局勢 升溫 庫德族 越境 攻打 伊朗 中東 局勢 升溫 庫德族 越境 攻打 伊朗 美國 聯手 以色列 伊朗 進行 重大 軍事 打擊 中東 地區 局勢 急遽 惡化 美國 以色列 數千 伊拉克 庫德族 武裝 人員 週三 越過 邊境...
2685,2026-03-07 14:14:29,Gossiping,Military,[新聞] 美軍4天轟2000目標嚇壞北京 專家：伊朗,美軍 天轟 目標 嚇壞 北京 專家 伊朗 中時 新聞網 許庭瑛 美軍 天轟 目標 嚇壞 北京 專家 伊朗 戰爭 反成 台灣 護盾 美國 伊朗 爆發 戰爭 外界 一度 憂心 美軍 武器 庫存 大量 消耗 削弱 美國 印太地 區的 防衛...
1162,2026-04-05 15:16:40,Gossiping,Military,[新聞] 「伊朗戰爭與我無關，為何是我承擔代價？,伊朗 戰爭 無關 承擔 代價 蘋果 日報 自由時報 參考 版規 下方 核准 名單 直接 官方 允許 風傳媒 記者 記者 名字 名字 外電 至少 法新社 李靖 寫出來 板規 伊朗 戰爭 無關 承擔 代價 巴基斯坦 油價 狂飆 萬人 生...


## 結論

1. `CountVectorizer + LogisticRegression` 可以作為基本示範；改用 `TF-IDF` 後，可以進一步比較不同模型的表現。
2. 在 5-fold CV 的比較下，SVM 與 Random Forest 通常會是表現較好的模型。
3. 係數分析顯示，`Military` 更偏向軍事情勢與戰況詞彙，`Gossiping` 則更偏向 PTT 問卦與社會討論語氣。
4. 若 `Military` 的 recall 偏低，代表模型仍容易把部分軍事板文章判成八卦板，可能原因是兩個版在伊朗戰爭主題上的新聞詞彙與討論語氣有所重疊。
